#

#Installations and Preperation

##Installing Packages

In [ ]:
!pip install -q pyvis


In [ ]:
!pip install -q von-sdk

##Importing Packages

In [ ]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
from geopy.geocoders import Nominatim
from time import sleep as wait
from tqdm import tqdm

import re
import os
import networkx as nx
import pandas as pd
import numpy as np
import math

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
import folium
from folium.plugins import MarkerCluster
import json
import shapely
import networkx as nx
import IPython
from pyvis.network import Network
import von


##Cloning

In [ ]:
%%bash
rm -rf LM4GeoAI Corpus2GeoRAG
git clone https://github.com/EtzionR/Corpus2GeoRAG.git

##Reading NER Data

In [ ]:
df = pd.read_parquet('Corpus2GeoRAG/ApplyNER/NER_output/post_NER_final.parquet')

print(f'Dataframe shape: {df.shape}\n\n')

df.head()


In [ ]:
df.entity_group.value_counts()

#GeoCoding

In [ ]:
loc_entities = list(df[df.entity_group=='LOC'].word.unique())

print(f'{len(loc_entities)} LOC entities found by the NER model\n')

In [ ]:
# Geocoding settings
TARGET_COUNTRIES = {'ua', 'ru'}             # preferred countries (ISO codes, as returned by Nominatim)
REGION_MARGIN = 0.15                        # max importance gap for preferring a UA/RU place / a city over the chosen match
SIMPLIFY_FACTOR = 0.002                     # simplification tolerance = factor * sqrt(area), i.e. ~0.2% of the place's size
SIMPLIFY_RANGE = (0.0005, 0.1)              # tolerance limits, in degrees (~50 m to ~10 km)
# All geocoded locations: the repo copy (cloned above), or the Google Drive copy when the repo does not have it
GEO_RESULTS_REPO_PATH = 'Corpus2GeoRAG/Graph_n_Geocoding/geocoding_results.parquet'
GEO_RESULTS_DRIVE_PATH = '/content/drive/Shareddrives/GIS/Omer/קורסים והשתלמויות/GeoAI/פרויקט גמר/geocoding_results.parquet'
ALLOW_FULL_GEOCODING = False  # True: geocode all LOC entities from scratch (~4 h) when neither copy is found
GEO_COLS = ['latitude', 'longitude', 'address', 'place_type', 'importance', 'geometry_wkt']

geolocator = Nominatim(user_agent="GeoAI_Wikipedia_Corpus_to_GeoGraph", timeout=10)

# Wrap geolocator to automatically delay requests (1 sec); errors are raised so they are not saved
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=False)

def get_importance(loc):
    return loc.raw.get('importance') or 0.0

def get_country_code(loc):
    return loc.raw.get('address', {}).get('country_code')

def geocode_entity(word):
    """Geocode one LOC entity, preferring a UA/RU place when it is within REGION_MARGIN of Nominatim's top match."""
    candidates = geocode(word, exactly_one=False, limit=5, language='en', addressdetails=True, geometry='wkt')
    if not candidates:
        return {'word': word, 'geo_reranked': False, 'geo_city_preferred': False}

    # Best UA/RU settlement or administrative area, considered only when the top match is outside UA/RU
    top = candidates[0]
    regional = [c for c in candidates
                if get_country_code(c) in TARGET_COUNTRIES and c.raw.get('class') in ('place', 'boundary')]
    contender = max(regional, key=get_importance) if regional and get_country_code(top) not in TARGET_COUNTRIES else None

    reranked = contender is not None and get_importance(contender) >= get_importance(top) - REGION_MARGIN
    loc = contender if reranked else top

    # Prefer a city/town named exactly like the entity over an oblast/state (e.g. 'Donetsk' -> city, not Donetsk Oblast)
    cities = [c for c in candidates
              if c.raw.get('addresstype') in ('city', 'town') and (c.raw.get('name') or '').lower() == word.lower()
              and get_country_code(c) == get_country_code(loc)]
    city = max(cities, key=get_importance) if cities and loc.raw.get('addresstype') == 'state' else None

    city_preferred = city is not None and get_importance(city) >= get_importance(loc) - REGION_MARGIN
    loc = city if city_preferred else loc

    return {
        'word': word,
        'latitude': loc.latitude,
        'longitude': loc.longitude,
        'address': loc.address,
        'place_type': loc.raw.get('addresstype') or loc.raw.get('type'),
        'importance': loc.raw.get('importance'),
        'geometry': fine_geometry(loc.raw.get('geotext')),  # polygon (or point, when OSM has no polygon), ~50 m
        # Audit fields (used by the validation and re-ranking cells, not added to the graph)
        'country_code': get_country_code(loc),
        'geo_reranked': reranked,
        'geo_city_preferred': city_preferred,
        'top_address': top.address,
        'top_importance': top.raw.get('importance'),
        'regional_address': contender.address if contender else None,
        'regional_importance': contender.raw.get('importance') if contender else None,
    }

def fine_geometry(wkt):
    """Polygon simplified at the smallest tolerance (SIMPLIFY_RANGE[0], ~50 m), as stored in the results file."""
    return shapely.to_wkt(shapely.from_wkt(wkt).simplify(SIMPLIFY_RANGE[0]), rounding_precision=5) if isinstance(wkt, str) else None

def simplify_geometry(wkt):
    """Simplify with a tolerance proportional to the geometry's size (sqrt of its area), clipped to SIMPLIFY_RANGE."""
    if not isinstance(wkt, str):
        return None, None
    geom = shapely.from_wkt(wkt)
    tolerance = min(max(SIMPLIFY_FACTOR * math.sqrt(geom.area), SIMPLIFY_RANGE[0]), SIMPLIFY_RANGE[1])
    return shapely.to_wkt(geom.simplify(tolerance), rounding_precision=5), tolerance


##Geocoding New Locations
Reads `geocoding_results.parquet` from the repo and sends only the LOC entities that are not in it yet to Nominatim (1 request/sec). When there are new ones, the file is updated: upload it to GitHub.

In [ ]:
import shutil

# Mount Google Drive when the repo does not have the results file (yet)
if not os.path.exists(GEO_RESULTS_REPO_PATH) and not os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Google Drive is not mounted ({type(e).__name__}): mount it to read the Drive copy')

# Geocoding results: the repo copy first, then the Drive copy
source_path = next((p for p in (GEO_RESULTS_REPO_PATH, GEO_RESULTS_DRIVE_PATH) if os.path.exists(p)), None)
if source_path is None and not ALLOW_FULL_GEOCODING:
    raise FileNotFoundError('geocoding_results.parquet was found neither in the repo nor on Google Drive. Mount Drive, '
                            'or set ALLOW_FULL_GEOCODING = True to geocode all LOC entities from scratch (~4 h)')
results = pd.read_parquet(source_path).to_dict('records') if source_path else []
print(f'Geocoding results: {source_path or "none, geocoding from scratch"}')

# Only LOC entities that are not in the results yet are sent to Nominatim
known = {r['word'] for r in results}
new_locations = [w for w in loc_entities if w not in known]
print(f'{len(loc_entities) - len(new_locations)} of {len(loc_entities)} LOC entities already geocoded, '
      f'{len(new_locations)} new (~{len(new_locations) / 3600:.1f} h at 1 request/sec)')

if new_locations:
    # The update is saved to every copy whose folder exists (repo and/or Drive)
    save_paths = [p for p in (GEO_RESULTS_REPO_PATH, GEO_RESULTS_DRIVE_PATH) if os.path.isdir(os.path.dirname(p))]
    for i, word in enumerate(tqdm(new_locations, desc='Geocoding new locations'), 1):
        try:
            results.append(geocode_entity(word))
        except Exception as e:
            tqdm.write(f'Skipped {word!r} ({type(e).__name__}), will retry on next run')
        if i % 100 == 0:  # checkpoint, fast compression
            pd.DataFrame(results).to_parquet(save_paths[0], index=False, compression='zstd')

    # Final save with strong compression (~2-3 min), small enough for GitHub, then copied to the other location
    pd.DataFrame(results).to_parquet(save_paths[0], index=False, compression='zstd', compression_level=19)
    for path in save_paths[1:]:
        shutil.copyfile(save_paths[0], path)
    print(f'Updated {len(results)} locations ({os.path.getsize(save_paths[0]) / 1e6:.1f} MB) in: {save_paths}. '
          'Upload the repo copy to GitHub before re-running the cloning cell')


##Geocoded Locations

In [ ]:
# Locations of this run: the stored ~50 m polygons are kept for the spatial INSIDE relations (geometry_fine)
# and simplified further by size for the graph (geometry_wkt)
geo_df = pd.DataFrame(results)
geo_df = geo_df[geo_df['word'].isin(loc_entities)].reset_index(drop=True)
geo_df['geometry_fine'] = geo_df.pop('geometry')
geo_df['geometry_wkt'], geo_df['geometry_tolerance'] = zip(*geo_df['geometry_fine'].map(simplify_geometry))

# Merge back into the main DataFrame (drop first, so re-running the cell does not duplicate columns)
df = df.drop(columns=GEO_COLS, errors='ignore').merge(geo_df[['word'] + GEO_COLS], on='word', how='left')


In [ ]:
# Example: the main DataFrame after geocoding, one row for each of the 10 most frequent LOC entities
loc_rows = df[df['entity_group'] == 'LOC']
top_locations = loc_rows['word'].value_counts().head(10).index

example = loc_rows.drop_duplicates('word').set_index('word').loc[
    top_locations, ['entity_group', 'Source', 'start', 'end', 'latitude', 'longitude', 'address', 'place_type', 'importance', 'geometry_wkt']]
example['geometry_wkt'] = example['geometry_wkt'].str[:45] + ' ...'

print(f"{loc_rows['latitude'].notna().sum()} of {len(loc_rows)} LOC rows have coordinates")
with pd.option_context('display.max_colwidth', 70):
    display(example)


In [ ]:

# Filter out entities without valid coordinates
map_data = geo_df.dropna(subset=['latitude', 'longitude'])

# Center map coordinates
center_lat = map_data['latitude'].mean()
center_lon = map_data['longitude'].mean()

# Esri World Street Map tiles
esri_tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}'
esri_attr = 'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China, TomTom'

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=3,
    tiles=esri_tiles,
    attr=esri_attr
)

# Add marker cluster
marker_cluster = MarkerCluster().add_to(m)

# Add markers with hover tooltips and click popups
for _, row in map_data.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=row['word'],
        popup=f"<b>Entity:</b> {row['word']}<br><b>Address:</b> {row['address']}"
              f"<br><b>Type:</b> {row['place_type']}<br><b>Importance:</b> {row['importance']:.2f}"
    ).add_to(marker_cluster)

# Display interactive map
m


##Polygon Map

In [ ]:
# Entities to draw: the 20 most frequent LOC entities, or set your own list, e.g. ['Donetsk', 'Kyiv', 'Crimea']
POLYGON_ENTITIES = df.loc[df['entity_group'] == 'LOC', 'word'].value_counts().head(20).index.tolist()

poly_data = geo_df[geo_df['word'].isin(POLYGON_ENTITIES) & geo_df['geometry_wkt'].notna()]

m_poly = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles=esri_tiles, attr=esri_attr)

# One toggleable layer per entity, drawn from the simplified geometry stored in the graph
for _, row in poly_data.iterrows():
    folium.GeoJson(
        shapely.from_wkt(row['geometry_wkt']),
        name=row['word'],
        tooltip=f"{row['word']} ({row['place_type']}) | tolerance {row['geometry_tolerance']:.4f}°",
        style_function=lambda _: {'color': '#d32f2f', 'weight': 2, 'fillOpacity': 0.1},
    ).add_to(m_poly)

folium.LayerControl(collapsed=False).add_to(m_poly)

# Display interactive map
m_poly


##Geocoding Validation

In [ ]:
# Nominatim returns WGS 84 geographic coordinates (EPSG:4326) in decimal degrees.
# Axis order: latitude/longitude columns and Folium use (lat, lon); WKT and shapely use (lon lat).
found = geo_df.dropna(subset=['latitude', 'longitude'])

in_range = found['latitude'].between(-90, 90) & found['longitude'].between(-180, 180)
# Simplification moves a boundary by at most its tolerance, so the point must lie within it
on_geometry = found.apply(
    lambda r: shapely.from_wkt(r['geometry_wkt']).distance(shapely.Point(r['longitude'], r['latitude'])) <= r['geometry_tolerance']
    if isinstance(r['geometry_wkt'], str) else True, axis=1)
outside_region = ~found['country_code'].isin(TARGET_COUNTRIES)
wkt_kb = found['geometry_wkt'].str.len() / 1024

print('CRS: WGS 84 (EPSG:4326), decimal degrees\n')
print(f'Geocoded: {len(found)}/{len(geo_df)} ({len(geo_df) - len(found)} not found)')
print(f'Coordinates out of range: {(~in_range).sum()}')
print(f'Point farther from its own geometry than the simplification tolerance: {(~on_geometry).sum()}')
print(f'Outside UA/RU: {outside_region.sum()}')
print(f"Simplified geometries: {wkt_kb.sum() / 1024:.1f} MB in total, largest {wkt_kb.max():.0f} KB ({found.loc[wkt_kb.idxmax(), 'word']})\n")
print(f"Not found (sample): {geo_df.loc[geo_df['latitude'].isna(), 'word'].head(20).tolist()}\n")

print('=== Place Types ===')
print(found['place_type'].value_counts().head(15))

# Rows failing a coordinate check
found[~in_range | ~on_geometry][['word', 'latitude', 'longitude', 'address']]


##Re-ranking Audit

In [ ]:
# 1. Entities whose top match was outside UA/RU while a UA/RU place was among the candidates
audit = geo_df.dropna(subset=['regional_address']).copy()
audit['importance_gap'] = (audit['top_importance'].astype(float) - audit['regional_importance'].astype(float)).round(3)
audit_cols = ['word', 'top_address', 'top_importance', 'regional_address', 'regional_importance', 'importance_gap']

# Sorted so the cases closest to the margin come first
reranked = audit[audit['geo_reranked']].sort_values('importance_gap', ascending=False)
kept = audit[~audit['geo_reranked']].sort_values('importance_gap')

# 2. Entities switched from an oblast/state to the city of the same name
city_preferred = geo_df[geo_df['geo_city_preferred']]

print(f'REGION_MARGIN = {REGION_MARGIN}: {len(reranked)} re-ranked to UA/RU, {len(kept)} kept the top match, '
      f'{len(city_preferred)} switched from oblast/state to city\n')

with pd.option_context('display.max_colwidth', None):
    print('=== Re-ranked to UA/RU (largest gaps first) ===')
    display(reranked[audit_cols].head(15))
    print('\n=== Kept Top Match (smallest gaps first) ===')
    display(kept[audit_cols].head(15))
    print('\n=== Oblast/State -> City ===')
    display(city_preferred[['word', 'top_address', 'top_importance', 'address', 'importance']].head(15))


#Knoledge Graph

##Von

In [ ]:
# Hugging Face token from Colab Secrets (key icon in the Colab sidebar -> secret named HF_TOKEN, notebook access on);
# falls back to a hidden prompt where Secrets are not available
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass('Hugging Face token: ')


In [ ]:
# Relation schemas per entity-type pair, in the canonical order PER -> ORG -> LOC ('a' = first entity, 'b' = second;
# for same-type pairs 'a' is the one mentioned first). Each choice: key -> (relation, subject, description), where
# subject is the entity the relation reads from ('a' / 'b', None = symmetric). Descriptions are filled with the real
# names, and both directions of a directional relation are separate choices, so von also decides the direction.
RELATION_SCHEMAS = {
    'PER_LOC': {
        'BORN_IN': ('BORN_IN', 'a', '{a} was born in, comes from, or is a native of {b}'),
        'LOCATED_IN': ('LOCATED_IN', 'a', '{a} lived, worked, served, fought, was held or was present in {b}'),
        'VISITED': ('VISITED', 'a', '{a} visited, travelled to or arrived in {b}'),
        'FLED_TO': ('FLED_TO', 'a', '{a} fled, escaped, emigrated or was deported to {b}'),
        'DIED_IN': ('DIED_IN', 'a', '{a} died, was killed or was buried in {b}'),
        'LEADS': ('LEADS', 'a', '{a} holds an office of {b}: its president, prime minister, minister, governor, mayor or head of administration'),
        'OPPOSES': ('OPPOSES', 'a', '{a} attacked, ordered an attack on, threatened or acted against {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} (its government or forces) attacked, killed, sanctioned, arrested, banned or accused {a}'),
        'STATED_ABOUT': ('STATED_ABOUT', 'a', '{a} is quoted as saying, claiming or writing something about {b}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'PER_ORG': {
        'MEMBER_OF': ('MEMBER_OF', 'a', '{a} is or was a member, employee, soldier, deputy, researcher, student or official of {b}'),
        'LEADS': ('LEADS', 'a', '{a} leads, heads, commands, chairs or founded {b}, or is its chief, director or deputy head'),
        'OPPOSES': ('OPPOSES', 'a', '{a} opposed, criticized, attacked or fought against {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} accused, criticized, sanctioned, arrested, detained, attacked or killed {a}'),
        'STATED_ABOUT': ('STATED_ABOUT', None, '{b} is cited as the source of a report or claim about {a} ("according to {b}"), or {a} is quoted about {b}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'PER_PER': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same person: another name, spelling, transliteration, title or nickname'),
        'RELATIVE_OF': ('RELATIVE_OF', None, '{a} and {b} are family: married, spouses, parent and child, or siblings'),
        'MET_WITH': ('MET_WITH', None, '{a} and {b} met, talked, phoned, negotiated or reached an agreement with each other'),
        'ALLIED_WITH': ('ALLIED_WITH', None, '{a} and {b} are allies, colleagues or confidants, or worked together or supported each other'),
        'OPPOSES': ('OPPOSES', 'a', '{a} opposed, criticized, attacked or accused {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} opposed, criticized, attacked or accused {a}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'ORG_LOC': {
        'BASED_IN': ('BASED_IN', 'a', '{a} is headquartered, based or stationed in {b}, has its capital in {b}, or was relocated to {b}'),
        'OPERATES_IN': ('OPERATES_IN', 'a', '{a} operates, fought, was deployed or works in {b}'),
        'ATTACKED': ('ATTACKED', 'a', '{a} attacked, shelled, bombed, invaded or struck {b}'),
        'CONTROLS': ('CONTROLS', 'a', '{a} controls, captured, occupied or governs {b}'),
        'SUPPORTS': ('SUPPORTS', None, 'one of {a} and {b} gave aid, weapons, funding or political support to the other'),
        'OPPOSES': ('OPPOSES', None, 'one of {a} and {b} sanctioned, banned, punished, accused or acted against the other'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'ORG_ORG': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same organization: an abbreviation, a former or later name, or {a} became or was renamed {b}'),
        'PART_OF': ('PART_OF', 'a', 'the sentence says {a} is a unit, branch, member or subsidiary of {b}, or is subordinated to {b}'),
        'HAS_PART': ('PART_OF', 'b', 'the sentence says {b} is a unit, branch, member or subsidiary of {a}, or is subordinated to {a}'),
        'ALLIED_WITH': ('ALLIED_WITH', None, '{a} and {b} cooperated, supported, funded or allied with each other'),
        'OPPOSES': ('OPPOSES', 'a', '{a} fought, attacked, accused, sanctioned or criticized {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} fought, attacked, accused, sanctioned or criticized {a}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'LOC_LOC': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same place: another name, spelling or former name'),
        'PART_OF': ('PART_OF', 'a', '{a} is located inside or is part of {b}'),
        'HAS_PART': ('PART_OF', 'b', '{b} is located inside or is part of {a}'),
        'ATTACKED': ('ATTACKED', 'a', '{a} (as a state or its forces) attacked, invaded, shelled or struck {b}'),
        'ATTACKED_BY': ('ATTACKED', 'b', '{b} (as a state or its forces) attacked, invaded, shelled or struck {a}'),
        'CONTROLS': ('CONTROLS', 'a', '{a} occupies, annexed or controls {b}, or part of {b} is controlled by {a}'),
        'CONTROLLED_BY': ('CONTROLS', 'b', '{b} occupies, annexed or controls {a}, or part of {a} is controlled by {b}'),
        'SUPPORTS': ('SUPPORTS', None, 'one of {a} and {b} (as states) gave aid, weapons, funding or diplomatic support to the other'),
        'NEGOTIATED_WITH': ('NEGOTIATED_WITH', None, '{a} and {b} (as states) held talks, negotiated or signed an agreement'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
}

TYPE_NAME = {'PER': 'person', 'ORG': 'organization', 'LOC': 'location'}

def classify_relation(a, b, type_a, type_b, sentence):
    """One von decision for the pair in its sentence -> (relation, subject entity or None, confidence)."""
    schema = RELATION_SCHEMAS[f'{type_a}_{type_b}']
    result = von.decide(
        state=sentence,
        choices={key: description.format(a=a, b=b) for key, (_, _, description) in schema.items()},
        instructions=f'Which relation between "{a}" ({TYPE_NAME[type_a]}) and "{b}" ({TYPE_NAME[type_b]}) does this sentence state explicitly? Choose CO_OCCURS_WITH if the sentence only mentions them together.'
    )
    relation, subject, _ = schema[result.choice]
    return relation, {'a': a, 'b': b, None: None}[subject], round(result.confidence, 2)


###Optimizing the Max_Relation_Distance

In [ ]:
# 1. Filter clean entities
clean_df = df[df['entity_group'].isin(['PER', 'LOC', 'ORG'])].copy()

pairs_data = []

# Group by paragraph
grouped = clean_df.groupby(['Source', 'Index'])

for (source, idx), group in grouped:
    entities = group[['word', 'entity_group', 'start', 'end']].drop_duplicates(subset=['word']).sort_values('start').to_dict('records')
    text = group['Text'].iloc[0]

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            e1, e2 = entities[i], entities[j]

            # Ensure e1 is the earlier entity in the text
            if e1['start'] > e2['start']:
                e1, e2 = e2, e1

            # Calculate character distance and extract text between them
            if e1['end'] <= e2['start']:
                dist = e2['start'] - e1['end']
                text_between = text[e1['end']:e2['start']]
            else:
                dist = 0
                text_between = "[OVERLAPPING]"

            # Extended Context Snippet: 40 chars before e1 -> 40 chars after e2
            start_bound = max(0, e1['start'] - 40)
            end_bound = min(len(text), e2['end'] + 40)
            context_snippet = text[start_bound:end_bound]

            pairs_data.append({
                'Source': source,
                'Paragraph Index': idx,
                'Entity 1': f"{e1['word']} ({e1['entity_group']})",
                'Entity 2': f"{e2['word']} ({e2['entity_group']})",
                'Distance (chars)': dist,
                'Text Between': text_between,
                'Context Snippet (±40 chars)': context_snippet,
                'Full Text': text
            })

            if len(pairs_data) >= 100:
                break
        if len(pairs_data) >= 100:
            break
    if len(pairs_data) >= 100:
        break

df_pairs = pd.DataFrame(pairs_data)

# Distance Binned Summary
bins = [0, 20, 40, 60, 80, 120, 200, 1000]
labels = ['0-20', '21-40', '41-60', '61-80', '81-120', '121-200', '200+']
df_pairs['Distance Bin'] = pd.cut(df_pairs['Distance (chars)'], bins=bins, labels=labels, include_lowest=True)

print(f"Sampled {len(df_pairs)} entity pairs across {df_pairs['Source'].nunique()} sources.\n")
print("=== Distance Bucket Distribution ===")
print(df_pairs['Distance Bin'].value_counts().sort_index())

In [ ]:
# Display sample rows including the new Context Snippet column
pd.set_option('display.max_colwidth', None)

# Sample up to 4 rows per distance bin
sample_inspection = df_pairs.sort_values('Distance Bin').groupby('Distance Bin', observed=False).head(4).reset_index(drop=True)

display_cols = [
    'Distance Bin',
    'Distance (chars)',
    'Entity 1',
    'Entity 2',
    'Text Between',
    'Context Snippet (±40 chars)'
]

print("=== Text & Context Inspection Between Entity Pairs ===")
sample_inspection[display_cols]

##Building the Graph - Nodes and Edges

###Text Index & Entity Pairs

In [ ]:
DECAY_LAMBDA = 0.01
MAX_RELATION_DISTANCE = 100  # 100-character distance gate
SENTENCE_END = ('.', '!', '?', '\n')
TYPE_ORDER = {'PER': 0, 'ORG': 1, 'LOC': 2}

clean_df = df[df['entity_group'].isin(['PER', 'LOC', 'ORG'])].copy()

# Text index: every unique text in the original order -> text_id 0..N-1
texts = df.drop_duplicates(['Source', 'Index'])[['Source', 'Index', 'Text', 'title', 'sub_title']].reset_index(drop=True)
text_id_of = {key: i for i, key in enumerate(zip(texts['Source'], texts['Index']))}
clean_df['text_id'] = [text_id_of[key] for key in zip(clean_df['Source'], clean_df['Index'])]

# One type per entity: the type the NER model assigned most often
entity_type = clean_df.groupby('word')['entity_group'].agg(lambda s: s.value_counts().index[0]).to_dict()

def gap(m1, m2, text):
    """Character distance d_k = max(0, start_2 - end_1) between two mentions, and the text between them."""
    (s1, e1), (s2, e2) = sorted([m1, m2])
    return (s2 - e1, text[e1:s2]) if e1 <= s2 else (0, '')

def is_comma_list(m1, m2, text, spans):
    """True when only commas separate two mentions once other entity mentions are removed (items of one list)."""
    (s1, e1), (s2, e2) = sorted([m1, m2])
    between = list(text[e1:s2])
    for s, e in spans:
        for k in range(max(s, e1), min(e, s2)):
            between[k - e1] = ' '
    return bool(re.fullmatch(r'[\s,;]*,[\s,;]*', ''.join(between)))

def sentence_of(text, start, end):
    """The sentence containing [start, end), cut at the same sentence-ending characters as the dual gate."""
    left = max(text.rfind(p, 0, start) for p in SENTENCE_END) + 1
    right = min([i for i in (text.find(p, end) for p in SENTENCE_END) if i != -1], default=len(text) - 1)
    return text[left:right + 1].strip()

# Occurrences of an entity pair in a text:
#  - one per sentence in which both entities pass the dual gate (closest mentions in that sentence) -> von.
#    von sees only the sentence and the two names, so more mention pairs in the same sentence would get the same answer
#  - if no sentence passes the gate, one CO_OCCURS_WITH occurrence (closest mentions in the text)
#  - two items of one comma list ("Horlivka, Mariupol, Yenakiieve") pass the gate but are flagged comma_list: no von
# The weight still counts one distance per text (primary): d_k of the closest mentions, preferring the gate
pair_rows = []
for (source, idx), group in tqdm(clean_df.groupby(['Source', 'Index'], sort=False), desc='Collecting entity pairs'):
    text = group['Text'].iloc[0]
    mentions = {w: list(zip(g['start'], g['end'])) for w, g in group.groupby('word', sort=False)}
    spans = list(zip(group['start'], group['end']))
    words = list(mentions)

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            best, per_sentence = None, {}
            for mu in mentions[words[i]]:
                for mv in mentions[words[j]]:
                    dist, between = gap(mu, mv, text)
                    gate = dist <= MAX_RELATION_DISTANCE and not any(p in between for p in SENTENCE_END)
                    if best is None or (not gate, dist) < (not best[3], best[2]):
                        best = (mu, mv, dist, gate)
                    if gate:
                        sentence_start = max(text.rfind(p, 0, min(mu[0], mv[0])) for p in SENTENCE_END)
                        if sentence_start not in per_sentence or dist < per_sentence[sentence_start][2]:
                            per_sentence[sentence_start] = (mu, mv, dist, gate)

            for mu, mv, dist, gate in (list(per_sentence.values()) or [best]):
                # Canonical order for the relation schemas: PER -> ORG -> LOC, same type -> first mentioned
                a, b = (words[i], mu), (words[j], mv)
                if (TYPE_ORDER[entity_type[b[0]]], b[1]) < (TYPE_ORDER[entity_type[a[0]]], a[1]):
                    a, b = b, a
                pair_rows.append((text_id_of[(source, idx)], source, a[0], b[0], int(a[1][0]), int(a[1][1]),
                                  int(b[1][0]), int(b[1][1]), dist, gate, (mu, mv) == best[:2],
                                  gate and is_comma_list(mu, mv, text, spans)))

pairs_df = pd.DataFrame(pair_rows, columns=['text_id', 'Source', 'a', 'b', 'a_start', 'a_end', 'b_start', 'b_end',
                                            'distance', 'gate', 'primary', 'comma_list'])
print(f"{len(texts)} texts, {len(entity_type)} entities, {pairs_df['primary'].sum()} entity pairs in texts, "
      f"{len(pairs_df)} occurrences, {pairs_df['gate'].sum()} pass the dual gate, "
      f"{pairs_df['comma_list'].sum()} of them comma lists -> {(pairs_df['gate'] & ~pairs_df['comma_list']).sum()} to von")


###Relation Classification (von)

In [ ]:
import hashlib

# von results are saved on Google Drive and reused in every run
VON_CACHE_PATH = '/content/drive/Shareddrives/GIS/Omer/קורסים והשתלמויות/GeoAI/פרויקט גמר/von_cache.parquet'
MIN_CONFIDENCE = 0.4  # weaker decisions fall back to CO_OCCURS_WITH (applied below, not in the cache)

if VON_CACHE_PATH.startswith('/content/drive') and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# von decides from the sentence and the two entities only, so a result is cached by (a, b, types, sentence) and reused
# for the same pair in the same sentence: in other texts, other runs and new corpus versions.
# SCHEMA_ID changes when RELATION_SCHEMAS changes, so results of an older schema are not reused
SCHEMA_ID = hashlib.md5(repr(RELATION_SCHEMAS).encode()).hexdigest()[:8]
KEY = ['a', 'b', 'type_a', 'type_b', 'sentence']

# Pairs that pass the dual gate and are not items of one comma list, with their sentence
gated = pairs_df[pairs_df['gate'] & ~pairs_df['comma_list']].copy()
gated['type_a'], gated['type_b'] = gated['a'].map(entity_type), gated['b'].map(entity_type)
gated['sentence'] = [sentence_of(texts.at[t, 'Text'], min(s1, s2), max(e1, e2))
                     for t, s1, e1, s2, e2 in zip(gated['text_id'], gated['a_start'], gated['a_end'], gated['b_start'], gated['b_end'])]

cache = pd.read_parquet(VON_CACHE_PATH) if os.path.exists(VON_CACHE_PATH) else pd.DataFrame(columns=KEY + ['schema'])
von_records = cache[cache['schema'] == SCHEMA_ID].to_dict('records')
done = {tuple(r[k] for k in KEY) for r in von_records}
todo = gated.drop_duplicates(KEY)
todo = todo[[key not in done for key in zip(*(todo[k] for k in KEY))]]
print(f'{len(gated)} pairs for von, {gated.drop_duplicates(KEY).shape[0]} unique (pair, sentence): '
      f'{len(von_records)} results in the cache, {len(todo)} to classify')

# Classify; the cache is saved every 2000 decisions and at the end
for n, row in enumerate(tqdm(todo.itertuples(index=False), total=len(todo), desc='Classifying relations (von)'), 1):
    try:
        relation, subject, confidence = classify_relation(row.a, row.b, row.type_a, row.type_b, row.sentence)
        von_records.append({'a': row.a, 'b': row.b, 'type_a': row.type_a, 'type_b': row.type_b, 'sentence': row.sentence,
                            'relation': relation, 'subject': subject, 'confidence': confidence, 'schema': SCHEMA_ID})
    except Exception as e:
        tqdm.write(f'Skipped {row.a!r} / {row.b!r} in text {row.text_id} ({type(e).__name__}), will retry on next run')
    if n % 2000 == 0 or n == len(todo):
        os.makedirs(os.path.dirname(VON_CACHE_PATH), exist_ok=True)
        pd.DataFrame(von_records).to_parquet(VON_CACHE_PATH, index=False)

# Attach the results: pairs that failed the gate, and decisions below MIN_CONFIDENCE, are CO_OCCURS_WITH
von_df = pd.DataFrame(von_records, columns=KEY + ['relation', 'subject', 'confidence'])
gated = gated.merge(von_df, on=KEY, how='left')
OCCURRENCE = ['text_id', 'a', 'b', 'a_start', 'b_start']
pairs_df = pairs_df.drop(columns=['relation', 'subject', 'confidence'], errors='ignore').merge(
    gated[OCCURRENCE + ['relation', 'subject', 'confidence']], on=OCCURRENCE, how='left')
weak = pairs_df['confidence'] < MIN_CONFIDENCE
pairs_df.loc[weak, 'relation'] = 'CO_OCCURS_WITH'
pairs_df.loc[weak, 'subject'] = None
pairs_df['relation'] = pairs_df['relation'].fillna('CO_OCCURS_WITH')
pairs_df[['subject', 'confidence']] = pairs_df[['subject', 'confidence']].astype(object).where(pairs_df[['subject', 'confidence']].notna(), None)

print(f"\n{weak.sum()} decisions below MIN_CONFIDENCE = {MIN_CONFIDENCE} set to CO_OCCURS_WITH\n")
print(pairs_df.loc[pairs_df['gate'], 'relation'].value_counts())


###Graph Assembly

In [ ]:
G = nx.Graph()

# Entity nodes; LOC nodes carry the geocoding fields
geo_attrs = geo_df.set_index('word')[GEO_COLS]
for entity, etype in entity_type.items():
    attrs = {'type': etype}
    if etype == 'LOC':
        values = geo_attrs.loc[entity] if entity in geo_attrs.index else {}
        attrs.update({col: (None if pd.isna(values.get(col)) else values.get(col)) for col in GEO_COLS})
    G.add_node(entity, **attrs)

# Text nodes, in the original order
for text_id, (source, index, text, title, sub_title) in enumerate(
        texts[['Source', 'Index', 'Text', 'title', 'sub_title']].itertuples(index=False, name=None)):
    G.add_node(f'TEXT_{text_id}', type='TEXT', text_id=text_id, text=text, source=source, index=int(index),
               title=None if pd.isna(title) else title, sub_title=None if pd.isna(sub_title) else sub_title)

# Entity-entity edges: one record per occurrence (pair in a sentence, or in a text when no sentence passes the gate).
# NetworkX exports an undirected edge with the earlier-added node as 'source', so spans are oriented the same way
node_order = {n: i for i, n in enumerate(G.nodes)}
for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df), desc='Building entity edges'):
    u, v, u_span, v_span = row.a, row.b, (row.a_start, row.a_end), (row.b_start, row.b_end)
    if node_order[u] > node_order[v]:
        u, v, u_span, v_span = v, u, v_span, u_span

    if not G.has_edge(u, v):
        G.add_edge(u, v, c_eff=0.0, sources=set(), relations={})
    data = G[u][v]
    if row.primary:  # C_eff counts one distance per text
        data['c_eff'] += math.exp(-DECAY_LAMBDA * row.distance)
    data['sources'].add(row.Source)
    data['relations'].setdefault(row.relation, []).append({
        'text_id': row.text_id, 'source_start': u_span[0], 'source_end': u_span[1],
        'target_start': v_span[0], 'target_end': v_span[1], 'subject': row.subject, 'confidence': row.confidence})

# W = max(0.1, round(S * ln(1 + C_eff), 2)); only weight and relations stay on the edge
for u, v, data in G.edges(data=True):
    data['weight'] = max(0.1, round(len(data.pop('sources')) * math.log1p(data.pop('c_eff')), 2))
n_entity_edges = G.number_of_edges()

# IN_THE_TEXT edges: entity -> text node, with every mention of the entity in that text
mentions_df = clean_df.drop_duplicates(['word', 'text_id', 'start'])
for (word, text_id), g in tqdm(mentions_df.groupby(['word', 'text_id'], sort=False), desc='Building IN_THE_TEXT edges'):
    G.add_edge(word, f'TEXT_{text_id}', weight=len(g),
               relations={'IN_THE_TEXT': [{'start': int(s), 'end': int(e)} for s, e in zip(g['start'], g['end'])]})

print(f"\nGraph created: {G.number_of_nodes()} nodes ({len(entity_type)} entities, {len(texts)} texts), "
      f"{n_entity_edges} entity edges, {G.number_of_edges() - n_entity_edges} IN_THE_TEXT edges.")


###Spatial Relations (INSIDE)

In [ ]:
# INSIDE: location A is inside location B when A's point lies in B's polygon.
# Only the smallest containing polygon is linked (direct parent), using the ~50 m geometry_fine to stay exact at borders
spatial = geo_df[geo_df['word'].map(entity_type) == 'LOC'].dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
geoms = [shapely.from_wkt(w) if isinstance(w, str) else None for w in spatial['geometry_fine']]
areas = np.array([g.area if g is not None else 0.0 for g in geoms])
area_idx = np.array([i for i, g in enumerate(geoms) if g is not None and g.geom_type in ('Polygon', 'MultiPolygon')])

tree = shapely.STRtree([geoms[i] for i in area_idx])
inner_idx, tree_idx = tree.query(shapely.points(spatial['longitude'], spatial['latitude']), predicate='within')

parent = {}
for i, j in zip(inner_idx, area_idx[tree_idx]):
    if i != j and areas[j] > areas[i] and (i not in parent or areas[j] < areas[parent[i]]):
        parent[i] = j

for i, j in parent.items():
    inner, outer = spatial.at[i, 'word'], spatial.at[j, 'word']
    record = {'subject': inner, 'basis': 'geometry'}
    if G.has_edge(inner, outer):
        G[inner][outer]['relations'].setdefault('INSIDE', []).append(record)
    else:
        G.add_edge(inner, outer, weight=0.1, relations={'INSIDE': [record]})  # no text: S = 0 -> W = max(0.1, 0) = 0.1

print(f'{len(parent)} INSIDE relations ({len(spatial)} located LOC entities, {len(area_idx)} with a polygon)')
pd.DataFrame([(spatial.at[i, 'word'], spatial.at[j, 'word']) for i, j in list(parent.items())[:10]], columns=['inside', 'parent'])


###Graph Inspection

In [ ]:
records = []

# One row per relation occurrence on entity-entity edges (IN_THE_TEXT edges are counted separately)
for u, v, data in G.edges(data=True):
    for rel, occurrences in data['relations'].items():
        if rel == 'IN_THE_TEXT':
            continue
        for occ in occurrences:
            records.append({'Entity 1': u, 'Entity 2': v, 'Relation': rel, 'Subject': occ['subject'],
                            'Confidence': occ.get('confidence'), 'Weight (W)': data['weight'], 'text_id': occ.get('text_id'),
                            'start': min(occ.get('source_start', 0), occ.get('target_start', 0)),
                            'end': max(occ.get('source_end', 0), occ.get('target_end', 0))})

df_relations = pd.DataFrame(records)

# 1. Relation distribution in the graph
print("=== Relation Distribution (occurrences) ===")
print(df_relations['Relation'].value_counts())
print(f"\nIN_THE_TEXT: {sum(len(d['relations']['IN_THE_TEXT']) for _, _, d in G.edges(data=True) if 'IN_THE_TEXT' in d['relations'])} mentions")
print("\n" + "="*50 + "\n")

# 2. Highest-confidence example of each text-based semantic relation, with its sentence
semantic = df_relations[~df_relations['Relation'].isin(['CO_OCCURS_WITH', 'INSIDE'])]
examples = semantic.sort_values('Confidence', ascending=False).groupby('Relation').head(3).sort_values(['Relation', 'Confidence'], ascending=[True, False])
examples['Sentence'] = [sentence_of(texts.at[t, 'Text'], s, e) for t, s, e in zip(examples['text_id'], examples['start'], examples['end'])]

pd.set_option('display.max_colwidth', None)
examples[['Relation', 'Entity 1', 'Entity 2', 'Subject', 'Confidence', 'Weight (W)', 'Sentence']].reset_index(drop=True)


In [ ]:
# Extract entity-entity edge attributes into a list (text nodes excluded)
edge_records = []
for u, v, data in G.edges(data=True):
    if 'IN_THE_TEXT' in data['relations']:
        continue
    edge_records.append({
        'Entity 1': u,
        'Entity 2': v,
        'Weight (W)': data['weight'],
        'Occurrences': sum(len(occ) for rel, occ in data['relations'].items() if rel != 'INSIDE'),
        'Relations': ', '.join(f'{rel} ({len(occ)})' for rel, occ in data['relations'].items())
    })

# Convert to DataFrame and sort by Weight
edges_df = pd.DataFrame(edge_records).sort_values(by='Weight (W)', ascending=False).reset_index(drop=True)

# Extract Top 5, Middle 5, and Lowest 5
n = len(edges_df)
top5 = edges_df.head(5).copy()
top5.insert(0, 'Category', 'Highest 5')

mid_idx = n // 2
mid5 = edges_df.iloc[max(0, mid_idx - 2) : min(n, mid_idx + 3)].head(5).copy()
mid5.insert(0, 'Category', 'Middle 5')

bot5 = edges_df.tail(5).copy()
bot5.insert(0, 'Category', 'Lowest 5')

# Combine into a single table
summary_table = pd.concat([top5, mid5, bot5], ignore_index=True)

print(f"Total Entity Edges Analyzed: {n}\n")
summary_table


### Proximity-Aware Composite Edge Weight Equation

To quantify the connection strength between any two entities ($u, v$) in the Knowledge Graph, we use a composite weight formula that integrates **character proximity**, **co-occurrence frequency**, and **source diversity**:

$$W(u, v) = \max\left(0.1,\ S \cdot \ln(1 + C_{\text{eff}})\right)$$

#### Variable Definitions:

* **$W(u, v)$ (Composite Weight):** The final numerical edge weight used to determine line thickness in PyVis and connection priority in the GraphRAG retriever.
* **$S$ (Source Diversity):** The number of unique document sources (`Source`) containing the connection between $u$ and $v$.
* **$C_{\text{eff}}$ (Effective Co-occurrence Count):** The sum of proximity-decayed scores across all $C$ texts in which $u$ and $v$ co-occur (one $d_k$ per text):
  $$C_{\text{eff}} = \sum_{k=1}^{C} e^{-\lambda \cdot d_k}$$
* **$d_k$ (Character Distance):** The distance in characters between the end offset of entity 1 and the start offset of entity 2 in occurrence $k$, measured on their closest mentions in the text (the closest pair inside one sentence when there is one).
* **$\lambda$ (Decay Rate):** Hyperparameter ($\lambda = 0.01$) controlling how rapidly connection strength decays with text distance.
* **$\text{Avg Proximity}$ (Average Proximity Score):** A normalized metric ($\text{Avg Proximity} = C_{\text{eff}} / C \in (0, 1]$) representing how close $u$ and $v$ appear to each other on average whenever co-occurring.

---

#### Rationale & Design Logic:

1. **Exponential Distance Decay ($e^{-\lambda \cdot d_k}$):**
   * Entities mentioned side-by-side in the same phrase (e.g., $d_k = 0 \implies e^0 = 1.0$) are far more likely to share a direct relation than entities separated by hundreds of characters in a long paragraph.
   * Applying exponential decay scales the value of each co-occurrence according to textual proximity.

2. **Strictly Positive Logarithmic Scaling ($\ln(1 + C_{\text{eff}})$):**
   * Using a natural logarithm enforces *diminishing returns* on repetition, preventing a single verbose document from artificially inflating an edge weight.
   * Using $\ln(1 + C_{\text{eff}})$ (or `math.log1p`) guarantees that any $C_{\text{eff}} > 0$ yields a strictly positive weight ($W > 0$), eliminating negative weights caused by small proximity scores.

3. **Linear Multiplier for Source Verification ($S$):**
   * Cross-validation across distinct document sources is the strongest indicator of factual reliability in GraphRAG.
   * Multiplying by $S$ heavily rewards connections corroborated across multiple independent articles.

---

#### Graph Schema:

* **Nodes**
  * `PER` / `ORG` / `LOC` entities (`type`); `LOC` nodes also hold `latitude`, `longitude`, `address`, `place_type`, `importance`, `geometry_wkt` (WGS 84, EPSG:4326).
  * `TEXT` nodes (`TEXT_<text_id>`): `text_id` (0..N-1, original order), `text`, `source`, `index`, `title`, `sub_title`.
* **Entity–entity edges:** `source`, `target`, `weight` ($W$ above) and `relations`, a dictionary of relation → list of occurrences:
  `{"text_id", "source_start", "source_end", "target_start", "target_end", "subject", "confidence"}`.
  An occurrence is the pair in one sentence that passes the dual gate (each sentence is classified separately), or, when no sentence of the text passes it, the pair in the text (`CO_OCCURS_WITH`). Two items of one comma list (only commas between them) are `CO_OCCURS_WITH` without von. $C_{\text{eff}}$ still counts one $d_k$ per text.
  Spans are character offsets of the source/target entity in the text node; `subject` is the entity the relation reads from (`null` = symmetric); `confidence` is von's score (`null` when the pair failed the dual gate).
* **`IN_THE_TEXT` edges:** entity → text node, `weight` = number of mentions, `relations = {"IN_THE_TEXT": [{"start", "end"}, ...]}`.
* **`INSIDE` relations:** location → smallest location whose polygon contains its point, `{"subject": <inner location>, "basis": "geometry"}`; without a text connection the edge weight is the floor $W = 0.1$ ($S = 0$).


##Exporting the Graph

In [ ]:
# 5. Export to RAG-compatible JSON format (no indentation: the graph holds the full corpus and is large)
KG_PATH = 'knowledge_graph.json'
graph_data = nx.node_link_data(G, edges="links")

with open(KG_PATH, 'w', encoding='utf-8') as f:
    json.dump(graph_data, f, ensure_ascii=False)

print(f"Successfully exported '{KG_PATH}' ({os.path.getsize(KG_PATH) / 1e6:.0f} MB).")


In [ ]:
# # ==========================================
# # 1. BUILD & EXPORT EDGES DATAFRAME
# # ==========================================
# edge_records = []

# for u, v, data in G.edges(data=True):
#     # Join list fields with a pipe '|' delimiter for clean CSV cell formatting
#     relations_str = "|".join(data.get('relations', []))
#     sources_str = "|".join(data.get('sources', []))
#     evidences_list = data.get('evidences', [])
#     primary_evidence = evidences_list[0] if evidences_list else ""

#     edge_records.append({
#         'source': u,
#         'target': v,
#         'source_type': G.nodes[u].get('type', ''),
#         'target_type': G.nodes[v].get('type', ''),
#         'weight': data.get('weight', 0.1),
#         'relations': relations_str,
#         'raw_occurrences': data.get('raw_occurrences', 0),
#         'effective_cooccurrences': round(data.get('c_eff', 0.0), 3),
#         'num_sources': len(data.get('sources', [])),
#         'sources': sources_str,
#         'primary_evidence': primary_evidence
#     })

# edges_df = pd.DataFrame(edge_records)

# # Sort by weight descending for easier inspection
# edges_df = edges_df.sort_values(by='weight', ascending=False).reset_index(drop=True)

# # Save to CSV using 'utf-8-sig' to ensure correct non-ASCII character encoding (e.g., Cyrillic/Hebrew in Excel)
# edges_df.to_csv('knowledge_graph_edges.csv', index=False, encoding='utf-8-sig')


# # ==========================================
# # 2. BUILD & EXPORT NODES DATAFRAME
# # ==========================================
# node_records = []

# for node, data in G.nodes(data=True):
#     node_records.append({
#         'entity': node,
#         'type': data.get('type', ''),
#         'latitude': data.get('latitude', None),
#         'longitude': data.get('longitude', None),
#         'address': data.get('address', None)
#     })

# nodes_df = pd.DataFrame(node_records)
# nodes_df.to_csv('knowledge_graph_nodes.csv', index=False, encoding='utf-8-sig')

# print(f"Exported {len(edges_df)} edges to 'knowledge_graph_edges.csv'")
# print(f"Exported {len(nodes_df)} nodes to 'knowledge_graph_nodes.csv'")



##Visulazing

In [ ]:
import IPython
from pyvis.network import Network

# The full graph is too large for the browser: draw the top entities by weighted degree (text nodes excluded)
TOP_N_ENTITIES = 150

entity_graph = G.subgraph(n for n, t in G.nodes(data='type') if t != 'TEXT')
top_nodes = sorted(entity_graph.degree(weight='weight'), key=lambda x: x[1], reverse=True)[:TOP_N_ENTITIES]
H = entity_graph.subgraph(n for n, _ in top_nodes)

# Initialize PyVis network (directed=False for undirected edges)
net = Network(
    notebook=True,
    cdn_resources='in_line',
    height='650px',
    width='100%',
    bgcolor='#1e1e1e',
    font_color='white',
    directed=False
)

color_map = {'PER': '#42a5f5', 'LOC': '#66bb6a', 'ORG': '#ffa726'}
pos = nx.spring_layout(H, seed=42, k=0.5)

# Add Nodes
for node, data in H.nodes(data=True):
    group_type = data.get('type', 'ORG')
    x_pos = float(pos[node][0]) * 1000
    y_pos = float(pos[node][1]) * 1000
    address = f"<br>Address: {data['address']}" if data.get('address') else ''

    net.add_node(
        node,
        label=node,
        color=color_map.get(group_type, '#cccccc'),
        x=x_pos,
        y=y_pos,
        title=f"Entity: {node}<br>Type: {group_type}{address}"
    )

# Add Edges
for u, v, data in H.edges(data=True):
    w = data.get('weight', 1.0)
    relations_str = ', '.join(f'{rel} ({len(occ)})' for rel, occ in data['relations'].items())

    edge_tooltip = (
        f"<b>Edge:</b> {u} &mdash; {v}<br>"
        f"<b>Relations:</b> {relations_str}<br>"
        f"<b>Composite Weight (W):</b> {w}"
    )

    net.add_edge(
        u, v,
        value=w,
        title=edge_tooltip
    )

# Freeze layout
net.set_options("""
var options = {
  "physics": {
    "enabled": true
  },
  "interaction": {
    "dragNodes": true,
    "zoomView": true
  }
}
""")

net.show('undirected_rag_graph.html')
IPython.display.HTML(filename='undirected_rag_graph.html')


##Metrics

In [ ]:
# Metrics are computed on the entity graph (text nodes and IN_THE_TEXT edges excluded)
entity_graph = G.subgraph(n for n, t in G.nodes(data='type') if t != 'TEXT').copy()

# Calculate degree centrality for all nodes in the entity graph
degrees = dict(entity_graph.degree())

# Sort entities by degree in descending order and get the top 10
top_10_energy_entities = sorted(degrees.items(), key=lambda item: item[1], reverse=True)[:10]

print("Top 10 highest energy (degree connectivity) entities:")
for idx, (entity, degree) in enumerate(top_10_energy_entities, 1):
    node_type = G.nodes[entity].get('type', 'Unknown')
    print(f"{idx}. {entity} ({node_type}) - Degree: {degree}")


In [ ]:
# Calculate betweenness centrality on the entity graph; exact betweenness takes hours at this size,
# so it is approximated from k sampled source nodes
BETWEENNESS_K = 500
betweenness = nx.betweenness_centrality(entity_graph, k=min(BETWEENNESS_K, entity_graph.number_of_nodes()), seed=42)

# Sort entities by betweenness centrality in descending order and get the top 10
top_10_betweenness_entities = sorted(betweenness.items(), key=lambda item: item[1], reverse=True)[:10]

print("Top 10 highest betweenness centrality entities:")
for idx, (entity, score) in enumerate(top_10_betweenness_entities, 1):
    node_type = G.nodes[entity].get('type', 'Unknown')
    print(f"{idx}. {entity} ({node_type}) - Betweenness Centrality: {score:.4f}")
